# Phase 5: Domain Discriminability Analysis

Goals:
- domain discriminability using a domain classifier
- AUC score (0.5=similar, 1.0=different)
- evaluate representation separability
- quantify how hard the domains are to tell apart

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import pickle
import json
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Path settings
ROOT = r'..'
FEAT_DIR = os.path.join(ROOT, 'features')
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 5: Domain Discriminability Analysis")
print("="*100)
print(f"Results: {RESULTS_DIR}\n")
np.random.seed(42)  # reproducible sampling

PHASE 5: Domain Discriminability Analysis
Results: ..\benchmark\results



In [2]:
# Load data
print("Loading data...\n")

with open(os.path.join(RESULTS_DIR, 'preprocessed_data.pkl'), 'rb') as f:
    phase1_data = pickle.load(f)

source_data = phase1_data['source_data']
target_data = phase1_data['target_data']
PROPERTIES = phase1_data['properties']
FOLD = phase1_data['fold']

# Load embeddings
emb_path = os.path.join(FEAT_DIR, 'graphormer_cv_embeddings.npz')
emb_data = np.load(emb_path, allow_pickle=True)
emb_keys = emb_data['keys']
emb_X = emb_data['X']
smiles_to_emb = {smi: emb_X[i] for i, smi in enumerate(emb_keys)}

print(f"Properties: {PROPERTIES}")
print(f"Embeddings: {len(smiles_to_emb)}\n")

Loading data...

Properties: ['clearance', 'half_life', 'fu']
Embeddings: 58304



In [3]:
# Domain Discriminability Analysis
domain_discriminability_results = {}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    domain_discriminability_results[prop] = {}
    
    # Extract embeddings
    src_df = source_data[prop]
    tgt_df = target_data[prop]
    
    X_src_list = []
    for idx, row in src_df.iterrows():
        if row['smiles_original'] in smiles_to_emb:
            X_src_list.append(smiles_to_emb[row['smiles_original']])
    
    X_tgt_list = []
    for idx, row in tgt_df.iterrows():
        if row['smiles_original'] in smiles_to_emb:
            X_tgt_list.append(smiles_to_emb[row['smiles_original']])
    
    if len(X_src_list) == 0 or len(X_tgt_list) == 0:
        print(f"  No embeddings available")
        continue
    
    X_src = np.array(X_src_list, dtype='float32')
    X_tgt = np.array(X_tgt_list, dtype='float32')
    
    print(f"  Source embeddings: {X_src.shape}")
    print(f"  Target embeddings: {X_tgt.shape}")
    
    # Sample if too large
    max_per_domain = 2000
    if len(X_src) > max_per_domain:
        idx = np.random.choice(len(X_src), max_per_domain, replace=False)
        X_src = X_src[idx]
    
    if len(X_tgt) > max_per_domain:
        idx = np.random.choice(len(X_tgt), max_per_domain, replace=False)
        X_tgt = X_tgt[idx]
    
    print(f"  (After sampling) Source: {X_src.shape}, Target: {X_tgt.shape}")
    
    # Combine data: label 0 = source, label 1 = target
    X_combined = np.vstack([X_src, X_tgt])
    y_combined = np.hstack([np.zeros(len(X_src)), np.ones(len(X_tgt))])
    
    print(f"  Combined dataset: {X_combined.shape}, {len(y_combined)} samples")
    
    # Train domain classifier 1: Logistic Regression
    print(f"\n  Training Logistic Regression classifier...")
    clf_lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
    
    # 5-fold CV
    auc_scores_lr = cross_val_score(clf_lr, X_combined, y_combined, 
                                     cv=5, scoring='roc_auc', n_jobs=-1)
    
    auc_mean_lr = np.mean(auc_scores_lr)
    auc_std_lr = np.std(auc_scores_lr)
    
    print(f"    LR AUC: {auc_mean_lr:.4f} ± {auc_std_lr:.4f}")
    print(f"    Fold scores: {[f'{s:.4f}' for s in auc_scores_lr]}")
    
    # Train domain classifier 2: Random Forest
    print(f"\n  Training Random Forest classifier...")
    clf_rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    
    auc_scores_rf = cross_val_score(clf_rf, X_combined, y_combined, 
                                     cv=5, scoring='roc_auc', n_jobs=-1)
    
    auc_mean_rf = np.mean(auc_scores_rf)
    auc_std_rf = np.std(auc_scores_rf)
    
    print(f"    RF AUC: {auc_mean_rf:.4f} ± {auc_std_rf:.4f}")
    print(f"    Fold scores: {[f'{s:.4f}' for s in auc_scores_rf]}")
    
    # Interpretation
    auc_avg = (auc_mean_lr + auc_mean_rf) / 2
    
    if auc_avg < 0.55:
        interpretation = "Similar domains (easy to transfer)"
    elif auc_avg < 0.70:
        interpretation = "Moderately different (partial transfer expected)"
    else:
        interpretation = "Very different domains (difficult transfer)"
    
    print(f"\n  Average AUC: {auc_avg:.4f}")
    print(f"  Interpretation: {interpretation}")
    
    domain_discriminability_results[prop] = {
        'n_source': int(len(X_src)),
        'n_target': int(len(X_tgt)),
        'lr_auc_mean': float(auc_mean_lr),
        'lr_auc_std': float(auc_std_lr),
        'rf_auc_mean': float(auc_mean_rf),
        'rf_auc_std': float(auc_std_rf),
        'avg_auc': float(auc_avg),
        'interpretation': interpretation,
    }


Property: CLEARANCE
  Source embeddings: (14386, 768)
  Target embeddings: (1027, 768)
  (After sampling) Source: (2000, 768), Target: (1027, 768)
  Combined dataset: (3027, 768), 3027 samples

  Training Logistic Regression classifier...
    LR AUC: 0.9566 ± 0.0093
    Fold scores: ['0.9515', '0.9620', '0.9711', '0.9545', '0.9438']

  Training Random Forest classifier...
    RF AUC: 0.9415 ± 0.0064
    Fold scores: ['0.9412', '0.9518', '0.9429', '0.9393', '0.9321']

  Average AUC: 0.9490
  Interpretation: Very different domains (difficult transfer)

Property: HALF_LIFE
  Source embeddings: (9202, 768)
  Target embeddings: (1164, 768)
  (After sampling) Source: (2000, 768), Target: (1164, 768)
  Combined dataset: (3164, 768), 3164 samples

  Training Logistic Regression classifier...
    LR AUC: 0.9566 ± 0.0088
    Fold scores: ['0.9646', '0.9418', '0.9637', '0.9515', '0.9616']

  Training Random Forest classifier...
    RF AUC: 0.9470 ± 0.0081
    Fold scores: ['0.9490', '0.9336', '0

In [4]:
# Summary Report
print(f"\n{'='*100}")
print("DOMAIN DISCRIMINABILITY SUMMARY")
print(f"{'='*100}\n")

print("AUC Interpretation:")
print("  AUC ≈ 0.5: Domains are similar (good for transfer)")
print("  AUC ≈ 1.0: Domains are very different (difficult transfer)")
print()

summary_data = []

for prop in PROPERTIES:
    if prop in domain_discriminability_results:
        res = domain_discriminability_results[prop]
        row = {
            'Property': prop,
            'LR_AUC': f"{res['lr_auc_mean']:.4f}",
            'RF_AUC': f"{res['rf_auc_mean']:.4f}",
            'Avg_AUC': f"{res['avg_auc']:.4f}",
            'Status': res['interpretation'],
        }
        summary_data.append(row)

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
else:
    print("No results")
print()


DOMAIN DISCRIMINABILITY SUMMARY

AUC Interpretation:
  AUC ≈ 0.5: Domains are similar (good for transfer)
  AUC ≈ 1.0: Domains are very different (difficult transfer)

 Property LR_AUC RF_AUC Avg_AUC                                      Status
clearance 0.9566 0.9415  0.9490 Very different domains (difficult transfer)
half_life 0.9566 0.9470  0.9518 Very different domains (difficult transfer)
       fu 0.9303 0.9371  0.9337 Very different domains (difficult transfer)



In [5]:
# Save results
with open(os.path.join(RESULTS_DIR, 'phase_05_domain_discriminability.json'), 'w') as f:
    json.dump(domain_discriminability_results, f, indent=2)

print("✓ Results saved to benchmark/results/")
print(f"  - phase_05_domain_discriminability.json")
print("\nPhase 5 Complete! Ready for Phase 6: Dataset Compatibility Analysis")

✓ Results saved to benchmark/results/
  - phase_05_domain_discriminability.json

Phase 5 Complete! Ready for Phase 6: Dataset Compatibility Analysis
